## Setup 1

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
# sample a document
doc = documents[0]
print(doc["content"])
print(doc["filename"])

# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple language model for that. It predicts 

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
# build the output schema of for the response of the llm
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
# load LLM client
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
# generate response for the sample document
import json

user_prompt = json.dumps(doc)
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt},
]
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

result = response.output_parsed
print(result)

questions=['What problem does retrieval-augmented generation solve for an LLM?', 'Why do we treat the language model like a black box in this course?', 'What are the main weaknesses of LLMs that this lesson talks about?', 'What are we building in this module to make RAG concrete?', 'What’s the difference between the first part of the module and the second part?']


In [7]:
# use llm_structured helper function to generate response
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [8]:
print(result, usage.input_tokens, usage.output_tokens)

questions=['What is the main goal of this module on retrieval-augmented generation?', 'Why does this course build the RAG system in plain Python instead of using a framework right away?', 'What does the lesson say are the main limits of large language models?', 'How does RAG help when the model doesn’t know an answer or can’t see your documents?', 'What will be covered in the first part of the module, and how is the second part different?'] 1020 105


In [9]:
# calculate price
from evaluation_utils import calc_price

cost = calc_price(usage)
cost

{'input_cost': 0.0007650000000000001,
 'output_cost': 0.00047250000000000005,
 'total_cost': 0.0012375}

## Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/02-environment.md
- 01-agentic-rag/lessons/03-rag.md

Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).

What's the average number of input tokens across these 3 calls?

- 140
- 1400
- 14000
- 140000

These numbers vary between runs, even with the same model, so pick the closest option. A different provider or model may land further apart, but the input tokens stay in the same order of magnitude - the prompt we send is the same.

Answer: 1400

In [10]:
# double check that the first 3 documents are the correct ones
[i['filename'] for i in documents[:3]]

['01-agentic-rag/lessons/01-intro.md',
 '01-agentic-rag/lessons/02-environment.md',
 '01-agentic-rag/lessons/03-rag.md']

In [11]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [12]:
# generate response for the target docs
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [13]:
ground_truth[:5]

[{'question': 'What is a Retrieval-Augmented Generation system, and why does it help when a model doesn’t already know the answer?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this module build the RAG setup in plain Python instead of starting with a framework?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main limits of an LLM that this lesson points out, like outdated knowledge or not seeing your files?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example app are they building in this module to show RAG in practice?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the first part of the module cover, and how is the second part different?',
  'document': '01-agentic-rag/lessons/01-intro.md'}]

In [14]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=119, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1139),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=89, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1375),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1858)]

In [15]:
# calculate the average input tokens from from usage
from evaluation_utils import calc_total_price

total_input = 0

for i in usages:
    total_input += i.input_tokens

avg_input = total_input / len(usages)
print(avg_input)

1353.0


## Setup 2

In [16]:
import pandas as pd

df_ground_truth = pd.read_csv('ground-truth.csv')
df_ground_truth.head()

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [17]:
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[:5]

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the course build in the first part of the module, and how is the second part different?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example app are you building here, and what data will it answer questions from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'}]

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [19]:
len(chunks)

295

In [20]:
# !uv run python download.py

In [21]:
# !uv add onnxruntime tokenizers numpy tqdm minsearch

In [22]:
# embed chunks for VectorSearch
from embedder import Embedder
import numpy as np

embed = Embedder()
batch_size = 50
X = []
texts = [chunk["content"] for chunk in chunks]

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [23]:
X.shape

(295, 384)

In [24]:
# build indices
from minsearch import VectorSearch, Index


tindex = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
tindex.fit(chunks)
vindex = VectorSearch(
    keyword_fields=["filename"]
)
vindex.fit(X, chunks)

def text_search(query, num_results=5):
    results = tindex.search(query, num_results=num_results)
    return results

def vector_search(query, num_results=5):
    vector_query = embed.encode(query)
    results = vindex.search(vector_query, num_results=num_results)

    return results

In [25]:
# test indices search functions
query = "How do I compute the cost of my LLM usage"
vquery = embed.encode(query)

tresult = text_search(query)
vresult = vector_search(query)

In [26]:
tresult[:2]

[{'start': 5000,
  'content': 'the first FAQ document.\n\n## Reusable utilities\n\nWe\'ll need this pattern again in other evaluation sections today, so\nwe put it in a reusable helper.\n\nIt contains helper functions we\'ll reuse in this module:\n\n- `llm_structured`: calls the OpenAI API with structured output\n- `llm_structured_retry`: retries structured-output calls when a\n  request fails\n- `calc_price`: calculates the price from token usage\n- `calc_total_price`: calculates the total price from multiple usage\n  objects\n- `map_progress`: runs work in parallel and tracks progress. We\'ll use it\n  in the next lesson.\n\nImport the structured-output helper:\n\n```python\nfrom evaluation_utils import llm_structured\n```\n\nUse it on the same document:\n\n```python\nresult, usage = llm_structured(\n    openai_client,\n    data_gen_instructions,\n    user_prompt,\n    Questions\n)\n\nprint(result.questions)\n```\n\n## Tracking cost\n\nThe response also contains token usage:\n\n```py

In [27]:
vresult[:2]

[{'start': 1000,
  'content': 'kens: int\n    response_time: float\n    cost: float\n    timestamp: datetime = field(default_factory=datetime.now)\n```\n\n## Cost calculation\n\nNext we need the cost of each call. The provider charges a price per\nmillion input tokens and another per million output tokens. So we\nmultiply each count by its rate and divide by a million. The `usage`\nobject comes straight from the LLM response. It carries the token counts\nfor the call we just made.\n\n```python\ndef calculate_cost(model, usage):\n    cost = 0\n    if "gpt-5.4-mini" in model:\n        cost = (usage.input_tokens * 0.15 + usage.output_tokens * 0.60) / 1_000_000\n    return cost\n```\n\nI keep copy-pasting a version of this function across modules, which\nisn\'t the tidiest thing in the world. For a real project you\'d pull it\ninto one shared place, but here it keeps each lesson self-contained.\n\n## Instrumented RAG\n\n`RAGBase` already works, and I like carrying it around as-is because i

In [28]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

Take the first question from the ground truth:

q = ground_truth[0]["question"]

After running text_search for it, what's the filename of the first result?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/03-rag.md
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/10-rag-next-steps.md

Answer: 01-agentic-rag/lessons/03-rag.md

In [29]:
print(ground_truth[0]["question"])

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [30]:
q = ground_truth[0]["question"]

tresult = text_search(q)
[r["filename"] for r in tresult]

['01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/01-intro.md']

## Q3. First result with vector search

After running vector_search for the same question, what's the filename of the first result?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/03-rag.md
- 04-evaluation/lessons/11-evaluation-intro.md
- 04-evaluation/lessons/12-rag-answers.md

This question was generated from 01-agentic-rag/lessons/01-intro.md. Notice that one method finds the right page at the top and the other doesn't. That's exactly why we measure across the whole dataset instead of trusting one query.

Answer: 01-agentic-rag/lessons/01-intro.md

In [31]:
vresult = vector_search(q)
[r["filename"] for r in vresult]

['01-agentic-rag/lessons/01-intro.md',
 '04-evaluation/lessons/11-evaluation-intro.md',
 '04-evaluation/lessons/12-rag-answers.md',
 '01-agentic-rag/lessons/10-rag-next-steps.md',
 '06-best-practices/lessons/01-intro.md']

## Setup 3

In [33]:
tresult[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [32]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [34]:
def compute_relevance(q, search_function, **kwargs):
    doc_id = q["filename"]
    results = search_function(query=q["question"], **kwargs)
    relevance = []

    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [36]:
# compute relevance total for a sample of ground truth docs
relevance_total_sample = compute_relevance_total(ground_truth[:3], text_search)
relevance_total_sample

  0%|          | 0/3 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1], [1, 0, 1, 0, 0], [1, 1, 0, 0, 1]]

In [37]:
# compute relevance total for all
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [38]:
len(relevance_total)

360

In [41]:
# compute hit rate
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt += 1

    return cnt / len(relevance)

In [43]:
hit_rate(relevance_total)

0.7583333333333333

In [44]:
# compute mean reciprocal rank (mrr)
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [45]:
mrr(relevance_total)

0.5942592592592594

In [46]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Q4. Evaluating text search

Evaluate text_search on the ground truth data.

What's the Hit Rate?
- 0.55
- 0.66
- 0.76
- 0.88

hit_rate is 0.758333, therefore the closest answer is 0.76.

Answer: 0.76

In [48]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

## Q5. Evaluating vector search

Now evaluate vector_search - the part we left for the homework, since the module only evaluated keyword search.

What's the MRR?
- 0.35
- 0.45
- 0.55
- 0.65

The MRR when using vector search is 0.548611, therefore the closest answer is 0.55.

Answer: 0.55

In [49]:
# evaluate hit_rate and mmr for the vector_search method
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

## Setup 4

In [50]:
# try hybrid search
evaluate(
    ground_truth,
    hybrid_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

In [60]:
k_values = [1, 50, 100, 200]
results = []

def hybrid_k_boost(query, k_value):
    return hybrid_search(query, k=k_value)

for k in k_values:
    result = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_k_boost(query, k)
    )
    result["k_value"] = k
    print(f"k_value = {k}: {result}")
    results.append(result)

  0%|          | 0/360 [00:00<?, ?it/s]

k_value = 1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449, 'k_value': 1}


  0%|          | 0/360 [00:00<?, ?it/s]

k_value = 50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667, 'k_value': 50}


  0%|          | 0/360 [00:00<?, ?it/s]

k_value = 100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667, 'k_value': 100}


  0%|          | 0/360 [00:00<?, ?it/s]

k_value = 200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667, 'k_value': 200}


In [61]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False)

,hit_rate,mrr,k_value
0,0.838889,0.648194,1
1,0.836111,0.637917,50
2,0.836111,0.637917,100
3,0.836111,0.637917,200


## Q6. Tuning hybrid search

The k constant in RRF controls how much the top ranks matter. A smaller k sharpens the gap between positions, so being at the top of a list counts for more. The RRF paper uses 60 as a default, but the best value depends on the data so let's measure it.

Evaluate hybrid_search over the full ground truth dataset for k values 1, 50, 100, and 200. Compare the MRR values for these runs.

Which k gives the best MRR?
- 1
- 50
- 100
- 200

Several values of k may give the same MRR. If there's a tie, pick the smallest k.

The best mrr result is 0.648194 which came from using k = 1.

Answer: 1